In [47]:
from heisenberg_hamiltonians import HeisenbergJ1J2, heisenberg_expr, heisenberg_expr_hadamard, heisenberg_expr_rot
from spin_lattices import KagomeLattice, SquareLattice, TriangularLattice, ChainLattice
from vmc_amplitude import get_csr_hamiltonian
import numpy as np
import matplotlib.pyplot as plt
from fourier_supervised_cleanroom import hadamard_transform
from fractions import Fraction

In [52]:
from numpy import hamming


lattice = KagomeLattice(2, 4)
J2 = 1
system = HeisenbergJ1J2(
    lattice=lattice, J1=1, J2=J2, use_symmetries=False, spin_inversion=None
)
system.get_eigenstates(1)

2023-11-03 17:18:54.472 | DEBUG    | heisenberg_hamiltonians:__init__:502 - number_spins=24
2023-11-03 17:18:54.473 | DEBUG    | heisenberg_hamiltonians:__init__:504 - Setting hamming_weight to half
2023-11-03 17:18:54.474 | DEBUG    | heisenberg_hamiltonians:__init__:515 - Symmetry group contains 0 elements
2023-11-03 17:18:54.475 | DEBUG    | heisenberg_hamiltonians:__init__:516 - Constructing basis


2023-11-03 17:18:54.535 | DEBUG    | heisenberg_hamiltonians:__init__:524 - Hilbert space dimension is 2704156
2023-11-03 17:18:54.545 | DEBUG    | heisenberg_hamiltonians:_find_cached_eigenstate:77 - Using cached version of eigenvalues / eigenstates from groundstates/HeisenbergJ1J2-KagomeLattice2x4-1.0-1.0-False-None-10.pickle
2023-11-03 17:18:54.969 | DEBUG    | heisenberg_hamiltonians:get_eigenstates:125 - Ground state energy is -43.0395889923


(array([-43.03958899, -42.82459918, -42.7457909 , -42.64435572,
        -42.638814  , -42.638814  , -42.56683915, -42.56683915,
        -42.56683915, -42.34783072]),
 array([[ 4.90411613e-09,  6.52025278e-08,  3.40441081e-08, ...,
          3.10850235e-09,  1.62077234e-08,  1.75996851e-08],
        [ 2.80245214e-08,  1.91775319e-08,  1.66782353e-10, ...,
          1.19297566e-09,  6.99751282e-08, -8.55468716e-08],
        [ 4.11419221e-19,  1.86579185e-18, -1.17569822e-08, ...,
         -1.36109137e-08,  2.42753447e-08, -4.70584705e-19],
        ...,
        [-2.25499309e-20, -1.76496527e-18, -1.17569822e-08, ...,
         -1.36109137e-08,  2.42753447e-08,  4.84731975e-19],
        [ 2.80245213e-08,  1.91775319e-08,  1.66782350e-10, ...,
          1.19297566e-09,  6.99751282e-08, -8.55468716e-08],
        [ 4.90411612e-09,  6.52025278e-08,  3.40441081e-08, ...,
          3.10850235e-09,  1.62077234e-08,  1.75996851e-08]]))

In [56]:
eps_train = 0.01
test_size = int(len(system.canonical_basis.states) * eps_train)
train_set = np.random.choice(system.canonical_basis.states, size=test_size, replace=False)
train_state = np.zeros(2 ** system.number_spins, dtype=np.float64)
ground_state = system.get_ground_state_in_full_basis()
train_state[train_set] = np.sign(ground_state[train_set])
reconstruction = hadamard_transform(train_state)

In [57]:
def overlap(x, y):
    return x @ y / np.linalg.norm(x) / np.linalg.norm(y)

In [58]:
overlap(reconstruction, ground_state)

0.030747765465775127

In [59]:
overlap(train_state, ground_state)

0.030747765465775127

In [67]:
overlap(np.abs(ground_state) * np.sign(reconstruction), ground_state)

0.41184267852299916